In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.model_selection import RandomizedSearchCV

pd.set_option("display.max_columns", None)
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r'../Dataset/processed_travel_data.csv')
df.head()

,booking_id,customer_id,route_id,ticket_price_gbp,seat_class,booking_channel,origin,destination,distance_km,route_category,customer_segment,loyalty_status,remaining_seats_realized,days_before_travel,booking_month,travel_month,is_weekend_travel,customer_quarterly_revenue,scarcity_index
0,B000000087,C000971,R0001,51.81,Standard,Web,Aberdeen,Leeds,264,Medium,Leisure,Not loyal,116,58,11,1,0,35.22,0.470905
1,B000000012,C004636,R0001,53.74,Standard,Web,Aberdeen,Leeds,264,Medium,Leisure,Gold,116,51,11,1,0,83.32,0.470905
2,B000000021,C004403,R0001,69.01,Flex,Web,Aberdeen,Leeds,264,Medium,Leisure,Not loyal,116,46,11,1,0,78.14,0.470905
3,B000000064,C010229,R0001,98.86,First,Mobile,Aberdeen,Leeds,264,Medium,Leisure,Not loyal,116,43,11,1,0,181.16,0.470905
4,B000000077,C009319,R0001,56.89,Standard,Mobile,Aberdeen,Leeds,264,Medium,Leisure,Not loyal,116,43,11,1,0,201.96,0.470905


In [12]:
df['seat_class'].value_counts()

seat_class
Standard    374168
Flex        114228
First        31317
Name: count, dtype: int64

In [14]:
df.groupby("seat_class")["ticket_price_gbp"].mean()

seat_class
First       95.971350
Flex        66.892387
Standard    54.814970
Name: ticket_price_gbp, dtype: float64

In [15]:
df.groupby("route_category")["ticket_price_gbp"].mean()

route_category
Long      95.962253
Medium    61.325856
Short     28.420587
Name: ticket_price_gbp, dtype: float64

#### seat_class and route_category are ordinal columns

- Standard < Flex < First for seat_class
- Short < Medium < Long for route_category

In [14]:
cat_features = df.select_dtypes(include=['object']).columns.to_list()[2:]
num_features = df.select_dtypes(include=['float', 'int']).columns.to_list()[1:]

features = cat_features + num_features

In [4]:
X = df[features]
y = df['ticket_price_gbp']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Training and Selecting a Base Model

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe_enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num_scale', StandardScaler(), num_features)
    ]
)

In [17]:
# base model selection

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

models = {
    'Linear_Regression': LinearRegression(),
    'Random_Forest': RandomForestRegressor(random_state=42),
    'Gradient_Boosting': GradientBoostingRegressor(random_state=42)
}

for name, model in models.items():
    pipeline = Pipeline(
    [
        ('preprocess', preprocessor),
        ('regressor', model)
    ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    print(f'>>> Evaluation for {name} <<<')
    print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
    print(f'r2_score: {r2_score(y_test, y_pred)}')
    print(f'MSE: {mean_squared_error(y_test, y_pred)}')
    print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}\n')

>>> Evaluation for Linear_Regression <<<
MAE: 11.193173409382124
r2_score: 0.7132227675427378
MSE: 263.3035790326453
RMSE: 16.226631783356808

>>> Evaluation for Random_Forest <<<
MAE: 3.1158328834324487
r2_score: 0.9637629607066716
MSE: 33.27091923485208
RMSE: 5.768094939826501

>>> Evaluation for Gradient_Boosting <<<
MAE: 9.034961628048087
r2_score: 0.8069378344757054
MSE: 177.25939651054918
RMSE: 13.313879844378542



In [6]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_cols = ['route_category', 'seat_class']
nominal_cols = [col for col in cat_features if col not in ordinal_cols]

ordinal_categories = [
    ['Short', 'Medium', 'Long'],  # route_category
    ['Standard', 'Flex', 'First']  # seat_class
]

rf_preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols)
    ],
    remainder='passthrough'
)

In [7]:
X_train_prepped = rf_preprocessor.fit_transform(X_train)
feature_names = rf_preprocessor.get_feature_names_out()

# Using a 50,000 row sample for feature selection
sample_size = min(50000, X_train_prepped.shape[0])

X_sample = X_train_prepped[:sample_size]
y_sample = y_train[:sample_size]

# Calculate Mutual Information scores once
mi_scores = mutual_info_regression(X_sample, y_sample, random_state=42)

# Identify indices for top 35 features
mi_results = pd.DataFrame({'Feature': feature_names, 'Score': mi_scores})
mi_results = mi_results.sort_values(by='Score', ascending=False)
top_35_indices = mi_results.head(35).index.tolist()
top_35_names = mi_results.head(35)['Feature'].tolist()

# Identify Top 35
mi_results = pd.DataFrame({'Feature': feature_names, 'Score': mi_scores})
mi_results = mi_results.sort_values(by='Score', ascending=False)
top_35_indices = mi_results.head(35).index.tolist()
top_35_names = mi_results.head(35)['Feature'].tolist()

In [ ]:
# Optimizing Rnadom Forest model
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt'],
    'max_samples': [0.5] 
}

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=10, 
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train using only the pre-selected top 35 columns
rf_search.fit(X_train_prepped[:, top_35_indices], y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, None],
                                        'max_features': ['sqrt'],
                                        'max_samples': [0.5],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 300, 500]},
                   random_state=42, scoring='neg_mean_absolute_error',
                   verbose=1)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def select_top_35(X):
    return X[:, top_35_indices]

final_pipeline = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('selector', FunctionTransformer(select_top_35)),
    ('model', rf_search.best_estimator_)
])

# Generate predictions for the test set
rf_y_pred = final_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, rf_y_pred)
mse = mean_squared_error(y_test, rf_y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, rf_y_pred)

In [ ]:
# setting mlflow experiment
mlflow.set_experiment('Voyage_ticket_price_prediction')
mlflow.set_tracking_uri('http://127.0.0.1:5000')

In [ ]:
# Logging with mlflow
with mlflow.start_run(run_name="RF_Final_Pipeline_Run"):
    # Log Best Hyperparameters
    mlflow.log_params(rf_search.best_params_)
    
    # Log Metrics
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)
    
    # Log the Full Pipeline for future use
    mlflow.sklearn.log_model(final_pipeline, "random_forest_pipeline")

print(f"Logged to MLflow. MAE: {mae:.4f}, R2: {r2:.4f}")

2026/02/26 21:02:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/26 21:02:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_Final_Pipeline_Run at: http://127.0.0.1:5000/#/experiments/1/runs/496b90d167ff4ceca57291f9eff327e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Logged to MLflow. MAE: 3.6339, R2: 0.9553
